# 39 智能优化算法

> **模块**：优化 ｜ **数据**：`../data/textbook/12.D 飞机侦察地理坐标数据.txt` ｜ **依赖**：numpy、pandas、matplotlib

## 1. 问题背景

模拟退火与遗传算法不依赖梯度，适合组合优化与多峰问题。前者以概率接受劣解跳出局部最优，后者通过选择、交叉、变异迭代种群。

## 2. 学习目标

- 理解退火温度与 Metropolis 接受准则
- 掌握遗传算法的编码、选择、交叉与变异
- 能调参并观察收敛过程

## 3. 环境与数据

- 依赖：numpy、pandas、matplotlib
- 数据：`../data/textbook/12.D 飞机侦察地理坐标数据.txt`

## 4. 方法与实现

### 4.1 模拟退火算法

- 优化函数(要使之最小)为: $f:x\rightarrow R^+$  
- 给定初始温度$T_0$和初始解$x(0)$, 并由$x_0$生成下一个解$x'\in N[x(0)]$,即$x_0$的邻域  
- 这个解是否被接受,依赖于以下概率: $$P(x(k)\rightarrow x')=\left\{\begin{array}{l}1,\qquad f(x')<f(x(k))\\e^{-\frac{f(x')-f(x(k))}{T_i}},\qquad otherwise.\\\end{array}\right.$$
- 这是一个马尔可夫过程

In [ ]:
# 课本例题

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# 数据读入, reshape
data = np.loadtxt('../data/textbook/12.D 飞机侦察地理坐标数据.txt')
data = data.reshape((data.shape[0]*4, 2))
data = np.insert(data, 0, values=[[70,40]],axis=0)
data = np.insert(data, data.shape[0], values=[[70,40]],axis=0)
# np.savetxt('飞机侦察地理坐标数据.txt',a,fmt='%0.4f')

# 地球半径
R = 6370

# 计算间距
d = np.zeros((data.shape[0], data.shape[0]), dtype=np.float)
tempdata = data/180*np.pi
for i in range(len(d)):
    for j in range(i+1, len(d)):
        d[i][j] = R*np.arccos(np.cos(tempdata[i][0]-tempdata[j][0]) * np.cos(tempdata[i][1]) * np.cos(tempdata[j][1])
                              + np.sin(tempdata[i][1]) * np.sin(tempdata[j][1]))
d = d + d.T

# 设置np.random的随机种子
np.random.seed(0)

# 先用蒙特卡洛算法求一个较好的初始解, 做1000次
length = np.inf
for times in range(1000):
    temppath = np.array([0], dtype=np.int)
    temppath = np.insert(temppath, 1, values=np.random.permutation([i+1 for i in range(100)]), axis=0)
    temppath = np.insert(temppath, temppath.shape[0], values=[101], axis=0)
    templen = 0
    for i in range(len(temppath)-1):
        templen += d[temppath[i]][temppath[i+1]]
    if templen < length:
        length = templen
        path = temppath
print('蒙特卡罗算法得到的路径长度: {}'.format(length), path, sep='\n')

# 下面用模拟退火算法
# 取降温系数alpha = 0.999, 终止温度e = 1e-30
alpha = 0.999
e = 1e-30
T = 1
for times in range(1000000):
    u, v = np.random.randint(low=1, high=101, size=(2))
    temppath = path.copy()
    temppath[u:v+1] = path[v:u-1:-1]
    # 计算路径长度差
    df = d[path[u-1]][path[v]] + d[path[u]][path[v+1]] - d[path[u-1]][path[u]] - d[path[v]][path[v+1]]
    # 尝试接受调整
    # 事实上, 不去接受增加长度的改变方式, 对结果几乎没有影响.
    # if df < 0:
    if df < 0 or np.exp(-df/T) >= np.random.rand():
        path = temppath
        length += df
    T = T * alpha
    if T < e:
        break
        
length = 0
for i in range(len(path)-1):
    length += d[path[i]][path[i+1]]
    
print('模拟退火算法得到的路径长度: {}, 花费时间: {}'.format(length, length/1000), path, sep='\n')

plt.figure(figsize=(20, 12))
divide = 1
coordinate_path = np.array([[data[i][0], data[i][1]] for i in path])
plt.scatter(coordinate_path[::divide, 0], coordinate_path[::divide, 1], marker='*', color='red')
plt.plot(coordinate_path[::divide, 0], coordinate_path[::divide, 1], 'b--')
    

### 4.2 遗传算法

In [ ]:
# 种群大小: M = 50
# 最大代数: G = 1000
# 交叉率: pc = 1, 交叉概率为1保证种群充分进化.
# 变异率: pm = 0.1, 一般而言, 变异发生的可能性较小
# 编码策略: 长为102的随机数列作为染色体, 值均在[0, 1]中, 按升序可以得到访问序列
# 初始种群可以用改良圈算法得到.
# 


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# 数据读入, reshape
data = np.loadtxt('../data/textbook/12.D 飞机侦察地理坐标数据.txt')
data = data.reshape((data.shape[0]*4, 2))
data = np.insert(data, 0, values=[[70,40]],axis=0)
data = np.insert(data, data.shape[0], values=[[70,40]],axis=0)
# np.savetxt('sj.txt',a,fmt='%0.4f')

# 地球半径
R = 6370

# 计算间距
d = np.zeros((data.shape[0], data.shape[0]), dtype=np.float)
tempdata = data/180*np.pi
for i in range(len(d)):
    for j in range(i+1, len(d)):
        d[i][j] = R*np.arccos(np.cos(tempdata[i][0]-tempdata[j][0]) * np.cos(tempdata[i][1]) * np.cos(tempdata[j][1])
                              + np.sin(tempdata[i][1]) * np.sin(tempdata[j][1]))
d = d + d.T

In [ ]:
np.random.seed(0)
# w为种群个数
w = 50
# g为进化的代数
g = 100

J = []
for k in range(w):
    # 生成初始解
    c1 = np.array([0, 101])
    c1 = np.insert(c1, 1, values=1 + np.random.permutation([i+1 for i in range(100)]), axis=0)
    for t in range(len(c1)):
        flag = 0
        for m in range(1, len(c1)-2):
            for n in range(m+2, len(c1)-1):
                if d[c1[m]][c1[n]] + d[c1[m+1]][c1[n+1]] < d[c1[m]][c1[m+1]] + d[c1[n]][c1[n+1]]:
                    c1[m:n+1] = c1[n:m-1:-1]
                    flag = 1
        if flag == 0:
            break
    J.append(c1)

np.savetxt('遗传算法初步结果.txt', J, fmt='%d')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)
# w为种群个数
w = 50
# g为进化的代数
g = 100

J = np.loadtxt('遗传算法初步结果.txt')
J = J / J.shape[1]
print(J.shape)
print(J)

size = J.shape[1]
# 遗传算法操作
for k in range(g):
    # 交配产生子代A的初始染色体
    A = J
    
    # 产生交叉操作的染色体对
    c = np.random.permutation([i for i in range(w)])
    for i in range(0, w, 2):
        F = int(1 + np.floor(100*np.random.rand(1)))
        temp = A[c[i], F:]
        A[c[i], F:] = A[c[i+1], F:]
        A[c[i+1], F:] = temp
        
    # 确定变异体
    by = []
    while not len(by):
        by = np.nonzero([np.random.rand() < 0.1 for i in range(w)])
    B = A[by]
    
    # 作变异
    for j in range(len(by)):
        bw = np.sort(1 + np.floor(100*np.random.rand(3))).astype(np.int)
        neworder = [i for i in range(bw[0])] + [i for i in range(bw[1], bw[2])] + [i for i in range(bw[0], bw[1])] + [i for i in range(bw[2], 102)]
        B[j, :] = B[j, neworder]
        
    # 父代,子代,变异体合起来
    G = np.concatenate((J, A, B), axis=0)
    
    # 恢复出序列
    SG = np.zeros(G.shape, dtype=np.int)
    for ind in range(G.shape[0]):
        SG[ind] = G[ind].argsort()
    
    # 计算各个个体代表的路径长度
    lengths = np.zeros(G.shape[0], dtype=np.float)
    for ind in range(len(SG)):
        for vex in range(len(SG[ind])-1):
            lengths[ind] += d[SG[ind][vex]][SG[ind][vex+1]]
    
    bestArg = lengths.argsort()
    J = G[bestArg[:50]]
      
print(lengths[bestArg[0]])
path = SG[0]
print(path)
plt.figure(figsize=(20, 12))
divide = 1
coordinate_path = np.array([[data[i][0], data[i][1]] for i in path])
plt.scatter(coordinate_path[::divide, 0], coordinate_path[::divide, 1], marker='*', color='red')
plt.plot(coordinate_path[::divide, 0], coordinate_path[::divide, 1], 'b--')
      

## 5. 小结

初温、降温系数、种群规模与变异率决定搜索效果；随机算法应多次运行并报告统计结果。

> 内容来源：教材参考实现《数学建模算法与应用》练习稿（MIT，Zhanghan Wang），原始文件已并入本篇，见 `THIRD-PARTY.md`。